# Chapter 12: Group Actions and Covering Maps

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 12, printed pp. 307-338, PDF pp. 325-356.

## Chapter Goal

This chapter turns covering spaces into quotients by symmetry. The main question is not just whether a map is a covering, but how much of a covering is encoded by a group action on the total space. The computational lesson below keeps three translations in view at the same time: deck transformations are symmetries that preserve the covering projection; a covering space action is exactly the local disjointness condition that makes an orbit map into a covering; and the universal cover organizes all other connected coverings through subgroups of the fundamental group.

The chapter has a useful rhythm. First, every covering map has an automorphism group, and normal coverings are exactly those whose automorphisms move transitively along each fiber. Second, if a group acts by homeomorphisms with small neighborhoods disjoint from their nontrivial translates, the quotient map is a normal covering and the acting group is the deck group. Third, when a universal cover exists, every subgroup of `pi_1(X)` produces an intermediate quotient, and conjugate subgroups describe the same unbased covering. Finally, for manifolds, local covering behavior is not enough: the quotient must also be Hausdorff, so properness enters as the geometric condition that keeps orbits from accumulating badly.

The visuals and checks in this notebook are not replicas of the textbook figures. They are small computational models designed to make the invariants inspectable: translations of the real line over the circle, disjoint sheets for a quotient action, a subgroup lattice model for torus coverings, a proof dependency graph for the classification theorem, and a hyperbolic polygon model for higher-genus surface quotients.

In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-12-group-actions-and-covering-maps/12-group-actions-and-covering-maps.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-12-group-actions-and-covering-maps/12-group-actions-and-covering-maps.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-12-group-actions-and-covering-maps/12-group-actions-and-covering-maps.ipynb",
  "notebook_title": "Chapter 12: Group Actions and Covering Maps",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import json
import math
import sys
import warnings

import numpy as np
import pandas as pd
import sympy as sp
from sympy.matrices.normalforms import hermite_normal_form, smith_normal_form

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
from IPython.display import display


def find_book_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'source_map.json').exists() and (candidate / 'utils').exists():
            return candidate
        nested = candidate / 'Introduction-to-Topological-Manifolds'
        if (nested / 'source_map.json').exists() and (nested / 'utils').exists():
            return nested
    raise RuntimeError('Could not locate the Introduction-to-Topological-Manifolds root.')


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.validation import image_stats, relative  # noqa: E402

warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame instead')

UNIT_KEY = 'chapter-12-group-actions-and-covering-maps'
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / 'figures'
HTML = ARTIFACT_ROOT / 'html'
CHECKS = ARTIFACT_ROOT / 'checks'
TABLES = ARTIFACT_ROOT / 'tables'


def notebook_artifact_path(path: Path) -> Path:
    return Path('..') / Path(path).resolve().relative_to(BOOK_ROOT.resolve())

SOURCE_SPAN = {
    'printed_pages': '307-338',
    'pdf_pages': '325-356',
    'pdftotext_note': 'Local pdftotext extraction was aligned with printed pages 307-338; the requested PDF span was also inspected for orientation.',
}

STORYBOARD = [
    {
        'concept': 'covering automorphisms of R -> S1',
        'representation': 'translation orbits over one fiber',
        'library': 'matplotlib + numpy',
        'artifact': 'figures/deck-translation-orbits.png',
        'inspection_target': 'points differing by integers project to the same point of the circle',
        'validation': 'exp(2*pi*i*(x+k)) equals exp(2*pi*i*x) to numerical tolerance and nonzero translations have no fixed sample points',
    },
    {
        'concept': 'covering space action condition',
        'representation': 'local interval and its disjoint translates plus proof dependency graph',
        'library': 'matplotlib + networkx',
        'artifact': 'figures/quotient-action-local-sheets.png and figures/classification-proof-graph.png',
        'inspection_target': 'small neighborhoods become separate sheets over the orbit quotient',
        'validation': 'all tested nontrivial translates of U are disjoint from U',
    },
    {
        'concept': 'classification of torus coverings by subgroups of Z^2',
        'representation': 'interactive lattice and subgroup fundamental parallelogram',
        'library': 'plotly + sympy + pandas',
        'artifact': 'html/torus-subgroup-covering-lattice.html and tables/torus-covering-subgroups.csv',
        'inspection_target': 'rank 0, rank 1, and rank 2 subgroups correspond to universal, cylinder, and finite torus coverings',
        'validation': 'Hermite/Smith normal forms and determinant index agree with the computed kernel size',
    },
    {
        'concept': 'proper free actions give manifold quotients',
        'representation': 'regular hyperbolic octagon with paired sides for genus 2 quotient',
        'library': 'numpy + matplotlib',
        'artifact': 'figures/hyperbolic-genus2-quotient-polygon.png',
        'inspection_target': 'edge-pairing transformations identify a polygonal fundamental region without local self-overlap',
        'validation': 'computed interior angle is pi/4 and adjacent hyperbolic edge lengths are equal within tolerance',
    },
]

storyboard_path = save_json(
    {'chapter': 'Chapter 12: Group Actions and Covering Maps', 'source_span': SOURCE_SPAN, 'items': STORYBOARD},
    CHECKS / 'visual-storyboard.json',
)

routing_rows = [
    {'concept': 'deck transformations', 'library': 'numpy, matplotlib', 'why': 'translations and fibers are low-dimensional and need a durable static diagram'},
    {'concept': 'quotient action proof flow', 'library': 'networkx, matplotlib', 'why': 'the theorem is a dependency chain of local disjointness, quotient openness, and sheet homeomorphisms'},
    {'concept': 'torus covering classification', 'library': 'plotly, sympy, pandas', 'why': 'lattices are inspectable interactively while subgroup normal forms and indices require exact integer algebra'},
    {'concept': 'surface quotients by proper actions', 'library': 'numpy, matplotlib', 'why': 'the Poincare disk polygon is a metric construction with checkable angles and edge lengths'},
]
routing_table_path = save_csv(routing_rows, TABLES / 'library-routing.csv')
display(pd.DataFrame(routing_rows))
display_artifact(notebook_artifact_path(storyboard_path))

## Computational Translation Guide

A covering map `q: E -> X` becomes computable once we separate three objects that are easy to confuse.

`Aut_q(E)` is the group of homeomorphisms of the total space that commute with `q`. In code we usually model these as explicit functions: integer translations of `R`, lattice translations of `R^2`, antipodal maps on a sphere, or edge-pairing transformations of a polygon. The invariant is simple: applying the symmetry before the projection gives the same point of the base as projecting first.

A covering space action is a group action whose nontrivial translates separate sufficiently small neighborhoods. Computationally, this is a collision test. Choose a neighborhood `U` around a point, push it by nearby group elements, and verify that the only translate meeting `U` is the identity translate. The theorem says that, under connected and locally path-connected hypotheses, this local test is exactly what is needed for `E -> E/Gamma` to be a normal covering.

The classification theorem translates subgroups into intermediate quotients. If `E` is the universal cover of `X`, then `pi_1(X)` acts by deck transformations on `E`; a subgroup `H` acts as a smaller symmetry group; and `E/H -> X` is the covering whose induced subgroup is `H`. The only ambiguity for unbased coverings is conjugacy, because moving the base point changes the subgroup by a conjugate.

For the torus, `pi_1(T^2)` is `Z^2`, so conjugacy disappears and subgroups are visible as integer lattices. Rank zero gives the universal cover `R^2 -> T^2`. Rank one gives a cylinder-like cover `S^1 x R -> T^2`. Rank two gives a finite torus cover, and the number of sheets is the index of the subgroup, computed as the absolute determinant of a basis matrix in Hermite normal form.

For manifolds as quotients, the extra computational word is proper. A free action can still have an orbit space with points that cannot be separated. Properness controls this by forcing only finitely much group motion to keep a compact test set near itself. For discrete groups acting on locally compact Hausdorff spaces, this makes the quotient Hausdorff; with freeness, the quotient of an `n`-manifold is again an `n`-manifold.

## Library Routing

The routing choice here follows the geometry rather than a single plotting habit. Matplotlib is used where the concept is a durable 2D proof picture: fibers over the circle, disjoint local sheets, and a hyperbolic disk polygon. NetworkX is used for the proof dependency graph because the classification theorem is best read as a directed chain of implications. Plotly is used for the torus lattice because learners benefit from panning and zooming the subgroup parallelogram and nearby integer points. SymPy is used only for exact integer algebra: Hermite normal form, Smith normal form, rank, determinant, and subgroup index. Pandas is used for small tables that compare subgroup type with the corresponding covering model.

The artifact contract for this chapter is: every visual has an inspection target, every saved check records the invariant being tested, and the final sanity cell asserts both file integrity and mathematical consistency. The saved storyboard lives at `artifacts/chapter-12-group-actions-and-covering-maps/checks/visual-storyboard.json`, so a later quality-control worker can compare the intended chapter plan with the generated notebook outputs.

## Visual Storyboard

The ordered storyboard below starts with the deck group that every learner can compute by hand, then moves to the quotient theorem, then to the classification theorem for the torus, and finally to the manifold quotient application. Each item has a file artifact and a check artifact, so the chapter's visual claims are inspectable rather than decorative.

In [ ]:
display(pd.DataFrame(STORYBOARD)[['concept', 'representation', 'library', 'artifact', 'validation']])
display_artifact(notebook_artifact_path(routing_table_path))
display_artifact(notebook_artifact_path(storyboard_path))

## Covering Automorphisms As Fiber Symmetries

The easiest deck group to see is the covering `epsilon: R -> S^1`, written computationally as `x |-> exp(2*pi*i*x)`. Integer translation does not change the image on the circle, so every map `T_k(x) = x + k` is a covering automorphism. The important point is not the formula itself, but what it proves about fibers. A fiber over a circle point is an integer-spaced orbit in the real line. The deck group acts freely because no nonidentity translation fixes a point, and it acts transitively on each fiber because any two lifts of the same circle point differ by an integer.

That transitivity is the model case for normal coverings. In a normal covering, automorphisms move any chosen lift of a base point to any other lift over the same base point. In a nonnormal covering, the automorphism group can be smaller than the whole fiber motion allowed by monodromy. The orbit criterion in the chapter says exactly when two fiber points can be related by a deck transformation: their induced subgroups inside the base fundamental group must match.

The figure below separates three layers. The circle records the base. The horizontal line records lifts in the universal cover. The highlighted points show one fiber. Arrows show several deck translations. The JSON check records the algebraic invariant that the projection is unchanged by every tested translation.

In [ ]:
theta = 0.18
fiber_points = np.array([theta + k for k in range(-2, 4)])
x_line = np.linspace(-2.25, 3.25, 500)
circle_t = np.linspace(0, 2 * np.pi, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.0, 1.7]})
ax0, ax1 = axes
ax0.plot(np.cos(circle_t), np.sin(circle_t), color='#264653', lw=2)
base_point = np.exp(2j * np.pi * theta)
ax0.scatter([base_point.real], [base_point.imag], s=80, color='#e76f51', zorder=3)
ax0.plot([0, base_point.real], [0, base_point.imag], color='#e76f51', lw=1.2, alpha=0.8)
ax0.set_aspect('equal')
ax0.set_title('base point on S1')
ax0.axis('off')

ax1.plot(x_line, np.zeros_like(x_line), color='#264653', lw=2)
ax1.scatter(fiber_points, np.zeros_like(fiber_points), s=60, color='#e76f51', zorder=3, label='one fiber')
for k in [-1, 1, 2]:
    start = theta
    end = theta + k
    ax1.annotate('', xy=(end, 0.18 + 0.07 * k), xytext=(start, 0.18 + 0.07 * k), arrowprops={'arrowstyle': '->', 'lw': 1.7, 'color': '#2a9d8f'})
    ax1.text((start + end) / 2, 0.29 + 0.07 * k, f'T_{k}', ha='center', color='#2a9d8f')
for point in fiber_points:
    ax1.plot([point, point], [-0.08, 0.08], color='#e9c46a', lw=1.2)
ax1.set_ylim(-0.35, 0.65)
ax1.set_yticks([])
ax1.set_xlabel('lift coordinate x in R')
ax1.set_title('deck translations preserve the projection')
ax1.spines[['left', 'right', 'top']].set_visible(False)
ax1.legend(loc='upper left')
fig.suptitle('Automorphisms of R -> S1 act freely and transitively on each fiber', y=1.02)
deck_png = save_matplotlib(fig, FIGURES / 'deck-translation-orbits.png')
plt.close(fig)

sample = np.linspace(-1.4, 1.4, 250)
translation_residuals = []
for k in range(-4, 5):
    before = np.exp(2j * np.pi * sample)
    after = np.exp(2j * np.pi * (sample + k))
    translation_residuals.append(float(np.max(np.abs(after - before))))
nonidentity_shift_distances = [abs(k) for k in range(-4, 5) if k != 0]
deck_checks = {
    'projection': 'epsilon(x)=exp(2*pi*i*x)',
    'tested_translations': list(range(-4, 5)),
    'max_projection_residual': max(translation_residuals),
    'minimum_nonidentity_point_motion': min(nonidentity_shift_distances),
    'fiber_points_for_theta': fiber_points.round(6).tolist(),
}
deck_checks_path = save_json(deck_checks, CHECKS / 'deck-translation-checks.json')
display_artifact(notebook_artifact_path(deck_png))
display_artifact(notebook_artifact_path(deck_checks_path))

## From Local Disjointness To A Quotient Covering

A group action by homeomorphisms is not automatically a covering action. The missing ingredient is local separation: each point must have a neighborhood whose nonidentity translates miss it. For the integer action on `R`, a small interval around a point has disjoint integer translates. The orbit space is a circle, and the quotient map is the familiar covering.

The covering space quotient theorem can be read as a proof pipeline. Local disjointness gives pairwise disjoint translates of a chosen path-connected neighborhood `U`. The quotient map sends all those translates to the same open set `V = q(U)`. Because the translates are disjoint, the restriction `q|_U` is injective; because quotient maps by group actions are open under the chapter hypotheses, `q|_U` is a homeomorphism onto `V`. Therefore `q^{-1}(V)` is split into sheets, and the orbit map is a normal covering.

This pipeline also explains why the acting group becomes the full deck group in the theorem. Every group element is already a deck transformation of the quotient map. Conversely, any deck transformation is determined by where it sends one point, and the orbit construction guarantees that the image point differs from the original by a unique group element.

In [ ]:
center = 0.37
radius = 0.22
shifts = np.arange(-3, 4)
intervals = [(center + k - radius, center + k + radius) for k in shifts]

fig, ax = plt.subplots(figsize=(10, 3.8))
for k, (left, right) in zip(shifts, intervals):
    color = '#2a9d8f' if k == 0 else '#8ab17d'
    ax.plot([left, right], [0, 0], lw=10, solid_capstyle='round', color=color, alpha=0.85)
    ax.text((left + right) / 2, 0.16, f'U+{k}' if k >= 0 else f'U{k}', ha='center', fontsize=9)
ax.scatter([center], [0], s=70, color='#e76f51', zorder=3)
for n in range(-3, 4):
    ax.axvline(n, color='#dddddd', lw=0.8, zorder=0)
ax.set_ylim(-0.45, 0.45)
ax.set_yticks([])
ax.set_xlabel('R with the action k.x = x + k')
ax.set_title('A small neighborhood and its integer translates are pairwise disjoint')
ax.spines[['left', 'right', 'top']].set_visible(False)
quotient_png = save_matplotlib(fig, FIGURES / 'quotient-action-local-sheets.png')
plt.close(fig)

def intervals_intersect(a, b):
    return max(a[0], b[0]) < min(a[1], b[1])

identity_interval = intervals[list(shifts).index(0)]
nontrivial_intersections = {
    int(k): bool(intervals_intersect(identity_interval, interval))
    for k, interval in zip(shifts, intervals)
    if k != 0
}

proof_graph = nx.DiGraph()
proof_edges = [
    ('covering space action', 'pairwise disjoint translates of U'),
    ('pairwise disjoint translates of U', 'q is injective on each sheet'),
    ('quotient map is open', 'q(U) is an open neighborhood'),
    ('q is injective on each sheet', 'q|U is a homeomorphism'),
    ('q(U) is an open neighborhood', 'q|U is a homeomorphism'),
    ('q|U is a homeomorphism', 'orbit map is a covering'),
    ('group acts transitively on orbits', 'covering is normal'),
    ('one-point determination of deck maps', 'deck group equals acting group'),
    ('orbit map is a covering', 'deck group equals acting group'),
]
proof_graph.add_edges_from(proof_edges)
fig, ax = plt.subplots(figsize=(11.5, 6.0))
pos = nx.spring_layout(proof_graph, seed=12, k=1.15)
nx.draw_networkx_edges(proof_graph, pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=14, edge_color='#6c757d', width=1.5)
nx.draw_networkx_nodes(proof_graph, pos, ax=ax, node_color='#f4a261', edgecolors='#264653', node_size=2300)
nx.draw_networkx_labels(proof_graph, pos, ax=ax, font_size=8)
ax.set_title('Proof scaffold: why a covering space action gives a normal quotient covering')
ax.axis('off')
proof_graph_png = save_matplotlib(fig, FIGURES / 'classification-proof-graph.png')
plt.close(fig)

quotient_checks = {
    'center': center,
    'radius': radius,
    'tested_shifts': shifts.astype(int).tolist(),
    'identity_interval': [float(identity_interval[0]), float(identity_interval[1])],
    'nontrivial_translate_intersections_with_U': nontrivial_intersections,
    'covering_space_action_condition_passed_for_test_window': not any(nontrivial_intersections.values()),
    'proof_graph_nodes': proof_graph.number_of_nodes(),
    'proof_graph_edges': proof_graph.number_of_edges(),
}
quotient_checks_path = save_json(quotient_checks, CHECKS / 'quotient-action-checks.json')
display_artifact(notebook_artifact_path(quotient_png))
display_artifact(notebook_artifact_path(proof_graph_png))
display_artifact(notebook_artifact_path(quotient_checks_path))

## Classification: Subgroups Produce Intermediate Coverings

The classification theorem says that if `X` has a universal cover, then connected coverings of `X` are controlled by subgroups of `pi_1(X)`, up to conjugacy. A subgroup `H` does not merely label a covering; it acts on the universal cover through the deck group. The quotient by that subgroup is the intermediate covering. This is the constructive half of the theorem: from algebra, build a space.

The torus is the cleanest computational laboratory because `pi_1(T^2) = Z^2` is abelian. There is no conjugacy ambiguity, and subgroups are integer lattices. The rank of the subgroup tells the shape of the covering total space. Rank zero means no deck identifications have been imposed on the universal cover, so the total space is `R^2`. Rank one identifies one primitive direction and leaves one unbounded direction, producing a cylinder-like cover. Rank two gives a finite-index sublattice of `Z^2`, so the quotient of `R^2` by that subgroup is another torus, and the number of sheets is the lattice index.

The interactive artifact uses the subgroup generated by columns of the matrix `A = [[3, 1], [0, 2]]`. The parallelogram spanned by those columns is a fundamental region for the subgroup inside `Z^2`; its area is `abs(det(A)) = 6`, so the associated finite torus cover has six sheets. Equivalently, the torus homomorphism induced by `A` has a six-point kernel.

In [ ]:
A = np.array([[3, 1], [0, 2]], dtype=int)
A_sp = sp.Matrix(A)
det_A = int(abs(A_sp.det()))
hnf_A = hermite_normal_form(A_sp)
snf_A = smith_normal_form(A_sp, domain=sp.ZZ)

def kernel_points_for_integer_torus_map(matrix: np.ndarray) -> list[list[float]]:
    det = int(round(abs(np.linalg.det(matrix))))
    inv = np.linalg.inv(matrix.astype(float))
    points = {}
    for m in product(range(det), repeat=2):
        x = (inv @ np.array(m, dtype=float)) % 1.0
        key = tuple(np.round(x, 10))
        points[key] = x
    return [points[key].round(10).tolist() for key in sorted(points)]

kernel_points = kernel_points_for_integer_torus_map(A)

coeffs = range(-2, 3)
integer_points = np.array([(i, j) for i in range(-4, 6) for j in range(-3, 5)])
subgroup_points = np.array([A @ np.array([i, j]) for i in coeffs for j in coeffs])
parallelogram = np.array([[0, 0], A[:, 0], A[:, 0] + A[:, 1], A[:, 1], [0, 0]], dtype=float)
unit_square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=float)
kernel_array = np.array(kernel_points)

fig = go.Figure()
fig.add_trace(go.Scatter(x=integer_points[:, 0], y=integer_points[:, 1], mode='markers', marker={'size': 5, 'color': '#adb5bd'}, name='Z^2 points'))
fig.add_trace(go.Scatter(x=subgroup_points[:, 0], y=subgroup_points[:, 1], mode='markers', marker={'size': 9, 'color': '#2a9d8f'}, name='subgroup generated by A'))
fig.add_trace(go.Scatter(x=parallelogram[:, 0], y=parallelogram[:, 1], mode='lines', line={'width': 4, 'color': '#e76f51'}, name='subgroup fundamental parallelogram'))
fig.add_trace(go.Scatter(x=unit_square[:, 0], y=unit_square[:, 1], mode='lines', line={'width': 3, 'dash': 'dash', 'color': '#264653'}, name='unit torus square'))
fig.add_trace(go.Scatter(x=kernel_array[:, 0], y=kernel_array[:, 1], mode='markers+text', text=[str(i) for i in range(len(kernel_array))], textposition='top center', marker={'size': 10, 'color': '#f4a261'}, name='kernel points mod Z^2'))
fig.update_layout(
    title='Torus covering from the subgroup A Z^2 inside Z^2',
    xaxis_title='first integer coordinate',
    yaxis_title='second integer coordinate',
    yaxis_scaleanchor='x',
    width=850,
    height=650,
    template='plotly_white',
)
torus_html = save_plotly_html(fig, HTML / 'torus-subgroup-covering-lattice.html')

torus_rows = [
    {'subgroup_rank': 0, 'canonical_subgroup': '{0}', 'covering_total_space': 'R^2', 'sheets': 'infinite', 'model': 'universal cover'},
    {'subgroup_rank': 1, 'canonical_subgroup': '<(2,1)>', 'covering_total_space': 'S^1 x R', 'sheets': 'infinite', 'model': 'cylinder cover of T^2'},
    {'subgroup_rank': 2, 'canonical_subgroup': '<(3,0),(1,2)>', 'covering_total_space': 'T^2', 'sheets': det_A, 'model': 'finite torus cover'},
]
torus_table_path = save_csv(torus_rows, TABLES / 'torus-covering-subgroups.csv')
torus_checks = {
    'matrix_A': A.tolist(),
    'determinant_index': det_A,
    'hermite_normal_form': np.array(hnf_A.tolist(), dtype=int).tolist(),
    'smith_normal_form': np.array(snf_A.tolist(), dtype=int).tolist(),
    'kernel_point_count': len(kernel_points),
    'kernel_points_mod_unit_square': kernel_points,
    'index_matches_kernel_size': det_A == len(kernel_points),
}
torus_checks_path = save_json(torus_checks, CHECKS / 'torus-covering-checks.json')
display_artifact(notebook_artifact_path(torus_html), height=620)
display(pd.DataFrame(torus_rows))
display_artifact(notebook_artifact_path(torus_checks_path))

## Applied Lab: Classify A Torus Cover From Generators

This lab turns the theorem into a small workflow. Start with a finite list of integer vectors in `Z^2`; treat them as generators for a subgroup `H`; compute the rank and a canonical Hermite normal form; then read off the covering model. The workflow is deliberately algebraic because the classification theorem is algebraic after the universal cover has been chosen.

The rank-zero case means `H` is trivial and no quotient of the universal cover has been taken. The rank-one case means exactly one independent deck direction has been divided out, leaving one unbounded coordinate. The rank-two case means `H` has finite index, so the cover has finitely many sheets. The exact sheet number is the index `[Z^2 : H]`, which is the determinant of the Hermite basis. Smith normal form refines the same information by showing the finite kernel as a product of cyclic factors.

Try changing the `examples` dictionary. For a rank-two subgroup, replacing the generators by another basis of the same subgroup should leave the Hermite normal form and sheet count unchanged. For a rank-one subgroup, multiplying a generator by `2` gives a different subgroup and therefore a different covering, even though it points in the same geometric direction.

In [ ]:
def classify_torus_subgroup(generators: list[tuple[int, int]]) -> dict[str, object]:
    if not generators:
        return {
            'generators': [],
            'rank': 0,
            'canonical_basis': [],
            'covering_model': 'R^2 -> T^2',
            'sheet_count': 'infinite',
            'normal_form': 'trivial subgroup',
        }
    matrix = sp.Matrix([[g[0] for g in generators], [g[1] for g in generators]])
    rank = int(matrix.rank())
    hnf = hermite_normal_form(matrix)
    if rank == 1:
        gen = [int(hnf[0, 0]), int(hnf[1, 0])]
        if gen[0] < 0 or (gen[0] == 0 and gen[1] < 0):
            gen = [-gen[0], -gen[1]]
        return {
            'generators': generators,
            'rank': 1,
            'canonical_basis': [gen],
            'covering_model': 'S^1 x R -> T^2',
            'sheet_count': 'infinite',
            'normal_form': f'<({gen[0]},{gen[1]})>',
        }
    if rank == 2:
        basis = sp.Matrix(hnf[:, :2])
        index = int(abs(basis.det()))
        snf = smith_normal_form(basis, domain=sp.ZZ)
        return {
            'generators': generators,
            'rank': 2,
            'canonical_basis': np.array(basis.tolist(), dtype=int).tolist(),
            'covering_model': 'T^2 -> T^2',
            'sheet_count': index,
            'normal_form': f'HNF={np.array(basis.tolist(), dtype=int).tolist()}, SNF={np.array(snf.tolist(), dtype=int).tolist()}',
        }
    raise ValueError('Subgroups of Z^2 can only have rank 0, 1, or 2.')


examples = {
    'universal': [],
    'slope cylinder': [(2, 1)],
    'same slope, larger subgroup step': [(4, 2), (6, 3)],
    'six sheet finite torus cover': [(3, 0), (1, 2)],
    'skew five sheet finite torus cover': [(2, 1), (1, 3)],
}
lab_results = []
for name, gens in examples.items():
    result = classify_torus_subgroup(gens)
    result['example'] = name
    lab_results.append(result)

lab_df = pd.DataFrame(lab_results)[['example', 'generators', 'rank', 'canonical_basis', 'covering_model', 'sheet_count', 'normal_form']]
lab_table_path = save_csv(lab_df.to_dict(orient='records'), TABLES / 'applied-lab-torus-subgroups.csv')
lab_checks_path = save_json({'examples': lab_results}, CHECKS / 'applied-lab-torus-subgroups.json')
display(lab_df)
display_artifact(notebook_artifact_path(lab_table_path))
display_artifact(notebook_artifact_path(lab_checks_path))

## Manifolds As Quotients: Why Properness Appears

A covering space action on a manifold gives a local covering model, but the quotient may still fail to be a manifold if it is not Hausdorff. The chapter isolates properness as the condition that prevents this failure for locally compact Hausdorff spaces. For a discrete group, properness can be tested by compact sets: only finitely many group elements may move a compact set so that it meets itself. This is the global companion to the local disjointness condition.

The surface application uses this idea in the hyperbolic disk. For a compact orientable surface of genus at least two, one can choose a regular hyperbolic polygon with `4g` sides and pair its sides by Mobius transformations. The generated group acts freely and properly on the disk, the polygon is a fundamental region, and the quotient is the surface. The disk is simply connected, so it becomes the universal cover.

The figure below builds the genus-two case: an octagon in the Poincare disk. Its eight sides are geodesic arcs meeting the unit circle orthogonally. The code numerically chooses the vertex radius so that each interior angle is `pi/4`, the angle needed for eight sectors to fit around a quotient vertex. The check does not prove the full Poincare polygon theorem, but it records the geometric data that the construction relies on: equal edge lengths, the target angle, and paired side labels.

In [ ]:
def hyperbolic_distance(z: complex, w: complex) -> float:
    numerator = 2 * abs(z - w) ** 2
    denominator = (1 - abs(z) ** 2) * (1 - abs(w) ** 2)
    return float(np.arccosh(1 + numerator / denominator))


def geodesic_arc(z1: complex, z2: complex, samples: int = 120) -> np.ndarray:
    system = np.array([[z1.real, z1.imag], [z2.real, z2.imag]], dtype=float)
    rhs = np.array([(abs(z1) ** 2 + 1) / 2, (abs(z2) ** 2 + 1) / 2], dtype=float)
    if abs(np.linalg.det(system)) < 1e-12:
        return np.linspace(z1, z2, samples)
    center_xy = np.linalg.solve(system, rhs)
    center = center_xy[0] + 1j * center_xy[1]
    radius = math.sqrt(abs(center) ** 2 - 1)
    theta1 = np.angle(z1 - center)
    theta2 = np.angle(z2 - center)
    delta_ccw = (theta2 - theta1) % (2 * np.pi)
    delta_cw = -((theta1 - theta2) % (2 * np.pi))
    candidates = []
    for delta in [delta_ccw, delta_cw]:
        theta = theta1 + np.linspace(0, delta, samples)
        points = center + radius * np.exp(1j * theta)
        candidates.append(points)
    candidates.sort(key=lambda pts: (np.max(np.abs(pts)) > 1 + 1e-7, np.mean(np.abs(pts))))
    return candidates[0]


def regular_vertices(radius: float, side_count: int) -> np.ndarray:
    angles = 2 * np.pi * np.arange(side_count) / side_count + np.pi / side_count
    return radius * np.exp(1j * angles)


def polygon_interior_angle(radius: float, side_count: int) -> float:
    vertices = regular_vertices(radius, side_count)
    previous_arc = geodesic_arc(vertices[-1], vertices[0], 180)
    next_arc = geodesic_arc(vertices[0], vertices[1], 180)
    v1 = previous_arc[-2] - vertices[0]
    v2 = next_arc[1] - vertices[0]
    cosine = np.dot([v1.real, v1.imag], [v2.real, v2.imag]) / (abs(v1) * abs(v2))
    return float(np.arccos(np.clip(cosine, -1, 1)))


genus = 2
side_count = 4 * genus
target_angle = np.pi / (2 * genus)
lo, hi = 0.02, 0.98
for _ in range(60):
    mid = (lo + hi) / 2
    if polygon_interior_angle(mid, side_count) > target_angle:
        lo = mid
    else:
        hi = mid
rho = (lo + hi) / 2
vertices = regular_vertices(rho, side_count)
edge_arcs = [geodesic_arc(vertices[i], vertices[(i + 1) % side_count], 150) for i in range(side_count)]
edge_lengths = [hyperbolic_distance(vertices[i], vertices[(i + 1) % side_count]) for i in range(side_count)]
angle = polygon_interior_angle(rho, side_count)
labels = ['a1', 'b1', 'a1^-1', 'b1^-1', 'a2', 'b2', 'a2^-1', 'b2^-1']

fig, ax = plt.subplots(figsize=(7.2, 7.2))
boundary = np.exp(1j * np.linspace(0, 2 * np.pi, 600))
ax.plot(boundary.real, boundary.imag, color='#264653', lw=2.0)
colors = ['#e76f51', '#2a9d8f', '#e76f51', '#2a9d8f', '#f4a261', '#457b9d', '#f4a261', '#457b9d']
for i, arc in enumerate(edge_arcs):
    ax.plot(arc.real, arc.imag, lw=3.0, color=colors[i])
    midpoint = arc[len(arc) // 2]
    ax.text(1.08 * midpoint.real, 1.08 * midpoint.imag, labels[i], ha='center', va='center', fontsize=10)
ax.scatter(vertices.real, vertices.imag, s=26, color='#1d3557', zorder=3)
ax.text(0, 0, 'genus 2\nfundamental octagon', ha='center', va='center', fontsize=11)
ax.set_aspect('equal')
ax.set_xlim(-1.12, 1.12)
ax.set_ylim(-1.12, 1.12)
ax.set_title('Hyperbolic polygon model for a surface quotient')
ax.axis('off')
hyperbolic_png = save_matplotlib(fig, FIGURES / 'hyperbolic-genus2-quotient-polygon.png')
plt.close(fig)

hyperbolic_checks = {
    'genus': genus,
    'side_count': side_count,
    'chosen_vertex_radius': float(rho),
    'target_interior_angle': float(target_angle),
    'computed_interior_angle': float(angle),
    'angle_error': float(abs(angle - target_angle)),
    'edge_lengths': [float(v) for v in edge_lengths],
    'max_edge_length_spread': float(max(edge_lengths) - min(edge_lengths)),
    'side_pairing_labels': labels,
}
hyperbolic_checks_path = save_json(hyperbolic_checks, CHECKS / 'hyperbolic-polygon-checks.json')
display_artifact(notebook_artifact_path(hyperbolic_png))
display_artifact(notebook_artifact_path(hyperbolic_checks_path))

## Proof And Invariant Scaffolds

The chapter's proof strategy can be summarized as a set of invariants that survive the passage between topology and algebra.

For automorphism groups, the invariant is the induced subgroup `q_* pi_1(E,e)` inside `pi_1(X,x)`. Moving the lift `e` around a fiber can conjugate this subgroup, and a deck transformation exists between two lifts exactly when the corresponding subgroups are equal. In the normal case equality holds across the whole fiber, so the deck group acts transitively. The structure theorem then identifies `Aut_q(E)` with `N_G(H)/H`, where `G = pi_1(X,x)` and `H = q_* pi_1(E,e)`. In the simply connected case, `H` is trivial and the deck group is the base fundamental group.

For quotient actions, the invariant is local sheet separation. If a small path-connected neighborhood has pairwise disjoint translates, then projecting it to the orbit space loses exactly the group labels and no local topology. This is why the quotient map is a covering and why it is normal: every fiber is a single group orbit.

For classification, the invariant is the subgroup up to conjugacy. Starting from a subgroup `H`, restrict the deck action of the universal cover to `H` and form `E/H`. Starting from a covering, recover the induced subgroup. These two moves invert each other up to the basepoint ambiguity. The torus lab makes the same statement concrete: Hermite normal form gives a stable representative for a subgroup of `Z^2`, and the determinant gives the sheet count for finite covers.

For manifold quotients, the invariant is separation of orbit closures. Properness turns compact tests into finite tests for discrete groups, which prevents distant group elements from producing inseparable quotient points. Free plus proper plus locally compact Hausdorff hypotheses give the manifold quotient theorem used in the surface application.

In [ ]:
required_artifacts = [
    storyboard_path,
    routing_table_path,
    deck_png,
    deck_checks_path,
    quotient_png,
    proof_graph_png,
    quotient_checks_path,
    torus_html,
    torus_table_path,
    torus_checks_path,
    lab_table_path,
    lab_checks_path,
    hyperbolic_png,
    hyperbolic_checks_path,
]
assert_artifacts(required_artifacts, min_bytes=64)

deck_data = json.loads(deck_checks_path.read_text(encoding='utf-8'))
quotient_data = json.loads(quotient_checks_path.read_text(encoding='utf-8'))
torus_data = json.loads(torus_checks_path.read_text(encoding='utf-8'))
hyperbolic_data = json.loads(hyperbolic_checks_path.read_text(encoding='utf-8'))
storyboard_data = json.loads(storyboard_path.read_text(encoding='utf-8'))

assert deck_data['max_projection_residual'] < 1e-12
assert deck_data['minimum_nonidentity_point_motion'] > 0
assert quotient_data['covering_space_action_condition_passed_for_test_window'] is True
assert torus_data['index_matches_kernel_size'] is True
assert torus_data['determinant_index'] == 6
assert hyperbolic_data['angle_error'] < 1e-10
assert hyperbolic_data['max_edge_length_spread'] < 1e-10
assert len(storyboard_data['items']) >= 4

png_stats = [image_stats(deck_png), image_stats(quotient_png), image_stats(proof_graph_png), image_stats(hyperbolic_png)]
assert all(item['max_channel_stddev'] > 1.0 for item in png_stats)

final_sanity = {
    'artifact_count_checked': len(required_artifacts),
    'png_stats': png_stats,
    'core_invariants': {
        'deck_projection_residual_lt_1e-12': deck_data['max_projection_residual'] < 1e-12,
        'quotient_local_disjointness': quotient_data['covering_space_action_condition_passed_for_test_window'],
        'torus_index_equals_kernel_size': torus_data['index_matches_kernel_size'],
        'hyperbolic_octagon_angle_target_met': hyperbolic_data['angle_error'] < 1e-10,
    },
    'book_relative_artifact_root': relative(ARTIFACT_ROOT, BOOK_ROOT),
}
final_sanity_path = save_json(final_sanity, CHECKS / 'final-sanity-summary.json')
assert_artifacts([final_sanity_path], min_bytes=64)
display(pd.DataFrame(png_stats))
display_artifact(notebook_artifact_path(final_sanity_path))

## Takeaways

Deck transformations are the symmetries of a covering that preserve the projection. They act freely on the total space, and for normal coverings they act transitively on each fiber.

A covering space action is a local disjointness condition, not just a free action. When the hypotheses of the quotient theorem hold, the orbit map `E -> E/Gamma` is a normal covering and the acting group is the full deck group.

The classification theorem is constructive. With a universal cover in hand, a subgroup of `pi_1(X)` acts on the universal cover; the quotient by that subgroup is the covering associated to it. Changing basepoints conjugates subgroups, so unbased isomorphism classes correspond to conjugacy classes.

For `T^2`, the theorem becomes integer lattice arithmetic. Rank zero, rank one, and rank two subgroups of `Z^2` give the universal, cylinder, and finite torus coverings. Hermite normal form makes the subgroup representative stable, and determinant gives the sheet count in the finite case.

For manifolds, local covering behavior must be paired with a Hausdorff quotient condition. Proper free actions of discrete groups on connected manifolds satisfy the needed separation condition and produce manifold quotients. Hyperbolic polygon side pairings are the chapter's major surface example: higher-genus compact orientable surfaces arise as quotients of the disk by properly acting discrete groups.